# Structure-augmented vs sequence-only isTPS, across 5 DPLM runs

For each of 5 DPLM checkpoints picked by **max median isTPS** from the
training-time `enzyme_explorer_validation/`, fold the 50 generated
sequences with ESMFold and rescore with EnzymeExplorer-with-structure;
compare against the matching sequence-only isTPS on the same 50 sequences.

Inputs are produced by `dplm/runs/structure_ee_batch.sbatch` on Karolina,
rsynced into each run's `enzyme_explorer_structure_validation/<step>/`.


In [ ]:
"""Compare sequence-only vs structure-augmented EnzymeExplorer isTPS on 5 DPLM runs.

Inputs (on pluskal.nas, populated by structure_ee_batch.sbatch + rsync):
  <run-folder>/enzyme_explorer_structure_validation/<step>/
      generated_sequences_enzyme_explorer.csv               (structure-EE, n=50)
      sequences_enzyme_explorer_sequence_only.csv           (seq-only on same 50)

Outputs (written to OUT_DIR):
  paired_per_run.png       — 5 panels, violin/box of seq-only vs structure
  bar_cross_run.png        — grouped bar chart of medians per run, both conditions
  summary.csv              — per-run medians/means + delta

Run as: jupytext-style (lines starting with `# %%` are cells) or `python structure_vs_sequence_compare.py`.
"""

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path("/Volumes/data/Users/Matous/terpene_synthases/output/dplm/training")
OUT_DIR = Path("/Volumes/data/Users/Matous/terpene_synthases/output/dplm/comparison/structure_vs_sequence")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [
    ("run_41_V",
     "TPS_dplm_150m_stage3_grid_run_41_lr1em3_wu2000_ts200000_ckpt10000_valee10000_lend0p0001_winit1em06_loratrue_ns50_r1_a2_ltmV",
     "step_70000"),
    ("run_41_LN_V",
     "TPS_dplm_150m_stage3_grid_run_41_LN_lr1em3_wu2000_ts200000_ckpt10000_valee10000_lend0p0001_winit1em06_loratrue_ns50_r1_a2_ltmV",
     "step_120000"),
    ("run_33_QVKO_lmhead",
     "TPS_dplm_150m_stage3_grid_run_33_lr1em4_wu200_ts20000_ckpt1000_valee1000_lend1em05_winit1em07_loratrue",
     "step_19000"),
    ("class_first_adapter_random",
     "TPS_dplm_150m_class_first_cyclization_grid_run_1_lr1em3_wu2000_ts200000_ckpt10000_valee10000_lend1em4_winit1em6_lorafalse_ns50_r1_a2_ltm0_adapter_random",
     "step_10000"),
    ("class_prepend_QVKO",
     "TPS_dplm_150m_class_prepend_first_cyclization_grid_run_1_lr1em3_wu2000_ts200000_ckpt10000_valee10000_lend1em4_winit1em6_loratrue_ns50_r1_a2_ltmQVKO",
     "step_10000"),
]


def find_col(cols, target):
    for c in cols:
        if c.strip().lower() == target.lower():
            return c
    return None


def load_pair(label, run_folder, step_name):
    base = DATA_ROOT / run_folder / "enzyme_explorer_structure_validation" / step_name
    struct_csv = base / "generated_sequences_enzyme_explorer.csv"
    seq_csv = base / "sequences_enzyme_explorer_sequence_only.csv"
    if not struct_csv.exists() or not seq_csv.exists():
        raise FileNotFoundError(f"{label}: missing {struct_csv} or {seq_csv}")
    s = pd.read_csv(struct_csv)
    q = pd.read_csv(seq_csv)
    s_id = find_col(s.columns, "ID")
    q_id = find_col(q.columns, "ID")
    s_tps = find_col(s.columns, "isTPS")
    q_tps = find_col(q.columns, "isTPS")
    s = s[[s_id, s_tps]].rename(columns={s_id: "ID", s_tps: "isTPS_structure"})
    q = q[[q_id, q_tps]].rename(columns={q_id: "ID", q_tps: "isTPS_seqonly"})
    merged = q.merge(s, on="ID", how="inner")
    merged["label"] = label
    merged["step"] = step_name
    return merged

In [ ]:
frames = [load_pair(label, folder, step) for label, folder, step in RUNS]
df = pd.concat(frames, ignore_index=True)
print(df.groupby("label").size())
print(df.head())

In [ ]:
# Summary table
def boot_ci(x, n=2000, q=(2.5, 97.5), seed=0):
    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n, len(x)), replace=True)
    medians = np.median(boots, axis=1)
    return np.percentile(medians, q)


rows = []
for label, _, step in RUNS:
    sub = df[df["label"] == label]
    so = sub["isTPS_seqonly"].to_numpy()
    st = sub["isTPS_structure"].to_numpy()
    so_lo, so_hi = boot_ci(so)
    st_lo, st_hi = boot_ci(st)
    rows.append({
        "label": label,
        "step": step,
        "n": len(sub),
        "seq_only_median": np.median(so),
        "seq_only_mean": np.mean(so),
        "seq_only_ci_lo": so_lo,
        "seq_only_ci_hi": so_hi,
        "structure_median": np.median(st),
        "structure_mean": np.mean(st),
        "structure_ci_lo": st_lo,
        "structure_ci_hi": st_hi,
        "delta_median": np.median(st) - np.median(so),
        "delta_mean": np.mean(st) - np.mean(so),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
print(summary[["label", "n", "seq_only_median", "structure_median", "delta_median",
               "seq_only_mean", "structure_mean", "delta_mean"]])

In [ ]:
# Figure 1 — paired per-run violins
labels = [r[0] for r in RUNS]
fig, axes = plt.subplots(1, len(labels), figsize=(3.0 * len(labels), 5), sharey=True)
for ax, label in zip(axes, labels):
    sub = df[df["label"] == label]
    so = sub["isTPS_seqonly"].to_numpy()
    st = sub["isTPS_structure"].to_numpy()
    parts = ax.violinplot([so, st], positions=[1, 2], showmedians=True, showextrema=False, widths=0.7)
    for pc, color in zip(parts["bodies"], ["#4C72B0", "#C44E52"]):
        pc.set_facecolor(color)
        pc.set_edgecolor("black")
        pc.set_alpha(0.7)
    ax.scatter([1] * len(so) + [2] * len(st), np.r_[so, st], s=8, color="black", alpha=0.4)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["seq-only", "+structure"])
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(label, fontsize=10)
    med_so = float(np.median(so))
    med_st = float(np.median(st))
    ax.text(0.5, -0.18,
            f"med {med_so:.3f} → {med_st:.3f}  (Δ {med_st-med_so:+.3f})",
            transform=ax.transAxes, ha="center", fontsize=9)
axes[0].set_ylabel("isTPS")
fig.suptitle("isTPS distribution: sequence-only vs ESMFold-structure-augmented EnzymeExplorer (n=50 each)", fontsize=12)
fig.tight_layout(rect=[0, 0.02, 1, 0.96])
out_paired = OUT_DIR / "paired_per_run.png"
fig.savefig(out_paired, dpi=200, bbox_inches="tight")
print(f"Saved {out_paired}")

In [ ]:
# Figure 2 — grouped bar chart, median per run with bootstrap CI
x = np.arange(len(summary))
w = 0.4
fig, ax = plt.subplots(figsize=(max(8, len(summary) * 1.4), 5))
so_med = summary["seq_only_median"].to_numpy()
st_med = summary["structure_median"].to_numpy()
so_err = np.vstack([so_med - summary["seq_only_ci_lo"], summary["seq_only_ci_hi"] - so_med])
st_err = np.vstack([st_med - summary["structure_ci_lo"], summary["structure_ci_hi"] - st_med])
ax.bar(x - w / 2, so_med, w, yerr=so_err, capsize=4, label="seq-only", color="#4C72B0")
ax.bar(x + w / 2, st_med, w, yerr=st_err, capsize=4, label="+structure", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(summary["label"], rotation=20, ha="right")
ax.set_ylabel("Median isTPS  (bootstrap 95% CI)")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
ax.set_title("Median isTPS per run: sequence-only vs +ESMFold structure")
for i, (s, t) in enumerate(zip(so_med, st_med)):
    ax.text(i - w / 2, s + 0.02, f"{s:.3f}", ha="center", fontsize=8)
    ax.text(i + w / 2, t + 0.02, f"{t:.3f}", ha="center", fontsize=8)
fig.tight_layout()
out_bar = OUT_DIR / "bar_cross_run.png"
fig.savefig(out_bar, dpi=200, bbox_inches="tight")
print(f"Saved {out_bar}")